# Mini-Wikipedia RAG Project

A retrieval-augmented generation (RAG) system built over the [rag-mini-wikipedia](https://huggingface.co/datasets/rag-datasets/rag-mini-wikipedia) dataset from HuggingFace. Core components — retrieval, chunking, reranking, BM25, evaluation — are **self-implemented from scratch** to demonstrate deep understanding of RAG internals.

**Self-implemented:**
- Recursive text chunking with configurable separators and overlap
- Direct FAISS vector index manipulation for retrieval
- Cross-encoder reranking with sentence-transformers
- BM25 scoring from scratch + Reciprocal Rank Fusion (RRF) for hybrid search
- Multi-metric evaluation framework (Recall@3, MRR, Token F1, LLM-as-Judge)

**Using LangChain for:** embedding wrapper (`OpenAIEmbeddings`), LLM wrapper (`ChatOpenAI`), FAISS docstore integration.

**Tech stack:** Python, FAISS, OpenAI `text-embedding-3-large`, `gpt-5.2`, `cross-encoder/ms-marco-MiniLM-L-6-v2`, HuggingFace Datasets

## 1. Setup & Configuration

In [1]:
import os
import getpass
"""
os.environ["LANGCHAIN_TRACING"] = "false"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass()
"""

'\nos.environ["LANGCHAIN_TRACING"] = "false"\nos.environ["LANGCHAIN_API_KEY"] = getpass.getpass()\n'

In [2]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OPENAI: ")

from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
model = ChatOpenAI(model="gpt-5.2")

## 2. Data Loading & Exploration

In [3]:
from datasets import load_dataset

corpus = load_dataset("rag-datasets/rag-mini-wikipedia", "text-corpus")
qa = load_dataset("rag-datasets/rag-mini-wikipedia", "question-answer")
print(f"Corpus: {corpus}\n")
print(f"Question and answer: {qa}")

Corpus: DatasetDict({
    passages: Dataset({
        features: ['passage', 'id'],
        num_rows: 3200
    })
})

Question and answer: DatasetDict({
    test: Dataset({
        features: ['question', 'answer', 'id'],
        num_rows: 918
    })
})


In [4]:
print(f"Example passage: {corpus['passages'][0]}\n")
print(f"Example Q&A: {qa['test'][:5]}")


Example passage: {'passage': 'Uruguay (official full name in  ; pron.  , Eastern Republic of  Uruguay) is a country located in the southeastern part of South America.  It is home to 3.3 million people, of which 1.7 million live in the capital Montevideo and its metropolitan area.', 'id': 0}

Example Q&A: {'question': ['Was Abraham Lincoln the sixteenth President of the United States?', 'Did Lincoln sign the National Banking Act of 1863?', 'Did his mother die of pneumonia?', "How many long was Lincoln's formal education?", 'When did Lincoln begin his political career?'], 'answer': ['yes', 'yes', 'no', '18 months', '1832'], 'id': [0, 2, 4, 6, 8]}


In [5]:
# Length distribution:
lengths = [len(p) for p in corpus["passages"]["passage"]]
print("Length distribution:")
print(f"Min: {min(lengths)}, Max: {max(lengths)}, Avg: {sum(lengths) // len(lengths)}")

Length distribution:
Min: 1, Max: 2515, Avg: 389


A minimum length of 1 character is suspicious — let's inspect these outliers:

In [6]:
for p in corpus["passages"]["passage"]:
    if len(p) == 1:
        print(p)

|


The single `"|"` character is a formatting artifact. Checking all passages under 10 characters:

In [7]:
short = [p for p in corpus["passages"]["passage"] if len(p) < 10]
print(f"Count: {len(short)}")
print(short)

Count: 39
['125px', '150px', 'johna.jpg', 'cited in', ' ;', 'Sumatra', 'Java', 'Sulawesi', 'Papua', '; Other', '|', '|}', '250px', '; Tourism', '-->', '  -->', '222px', '#Oulu', '#Lapland', '#Ã\x85land', '|-', '|-', ' -->', ' }', 'with.', '*  .', 'tags-->', ';History', ';Economy', ';Language', ';Culture', ';Other', '100px', '#Al Khawr', '#Mesaieed', '----', 'Travel', 'Maps', 'Overviews']


In [8]:
lengths = [len(p) for p in corpus["passages"]["passage"] if len(p) >= 10]
print(f"Total passages after filtering: {len(lengths)}")
print(f"Under 500: {sum(1 for l in lengths if l < 500)}")
print(f"500-1000: {sum(1 for l in lengths if 500 <= l < 1000)}")
print(f"Over 1000: {sum(1 for l in lengths if l >= 1000)}")

Total passages after filtering: 3161
Under 500: 2153
500-1000: 814
Over 1000: 194


Most passages (68%) are already under 500 characters, and only 194 exceed 1,000. The 39 passages under 10 characters are formatting artifacts (HTML tags, separators, stubs) and will be filtered out.

After experimenting with different chunk sizes (see Section 8.3 below), **500/50** was selected for the best accuracy (90%) with a reasonable chunk count (4,658).

## 3. Text Preprocessing & Chunking

In [9]:
passages = corpus["passages"]
filtered = [
    {"text": p["passage"], "id": p["id"]}
    for p in passages if len(p["passage"]) >= 10
]
print(f"Before: {len(passages)}, After: {len(filtered)}")

Before: 3200, After: 3161


### Custom Recursive Text Splitter

A recursive text splitter that mirrors the logic of LangChain's `RecursiveCharacterTextSplitter`, implemented from scratch:

1. Tries separators in priority order: `\n\n` → `\n` → `. ` → ` ` → `""`
2. Splits on the first separator found in the text
3. Merges pieces until they exceed `chunk_size`
4. Preserves `chunk_overlap` characters between consecutive chunks for context continuity
5. Recursively re-splits any oversized chunk using finer-grained separators

In [10]:
def chunk_text(text, chunk_size=500, chunk_overlap=50, separators=None):
    if separators is None:
        separators = ["\n\n", "\n", ". ", " ", ""]
    
    if len(text) <= chunk_size:
        return [text]

    sep = ""
    for s in separators:
        if s in text or s == "":
            sep = s
            break
    # Find the best separator, if text doesn't have \n\n, skip to next one

    if sep == "":
        pieces = list(text) # ["h", "e", "l", "l", "o"]
    else:
        pieces = text.split(sep)

    chunks = []
    current = pieces[0] 

    for piece in pieces[1:]:
        combine = current + sep + piece
        if len(combine) <= chunk_size:
            current = combine
        else:
            chunks.append(current)

            if chunk_overlap > 0 and len(current) > chunk_overlap:
                overlap_text = current[-chunk_overlap:]
                current = overlap_text + sep + piece
            else:
                current = piece

    chunks.append(current)

    remaining_seps = separators[separators.index(sep) + 1:] if sep in separators else separators[1:]
    final_chunks = []

    for chunk in chunks:
        if len(chunk) > chunk_size and remaining_seps:
            final_chunks.extend(chunk_text(chunk, chunk_size, chunk_overlap, remaining_seps))
        else:
            final_chunks.append(chunk)

    return final_chunks

In [11]:
print(chunk_text("Hello world"))
long_passage = corpus["passages"][0]["passage"]
for i, chunk in enumerate(chunk_text(long_passage)):
    print(f"Chunk {i}: ({len(chunk)} chars) {chunk[:80]}...")

['Hello world']
Chunk 0: (250 chars) Uruguay (official full name in  ; pron.  , Eastern Republic of  Uruguay) is a co...


In [12]:
# Build documents with best config: chunk_size=500, chunk_overlap=50
documents = []
for item in filtered:
    chunks = chunk_text(item["text"], chunk_size=500, chunk_overlap=50)
    for i, chunk in enumerate(chunks):
        documents.append({
            "text": chunk,
            "passage_id": item["id"],
            "chunk_index": i
        })

print(f"Total chunks: {len(documents)}")
print(documents[0])

Total chunks: 4658
{'text': 'Uruguay (official full name in  ; pron.  , Eastern Republic of  Uruguay) is a country located in the southeastern part of South America.  It is home to 3.3 million people, of which 1.7 million live in the capital Montevideo and its metropolitan area.', 'passage_id': 0, 'chunk_index': 0}


## 4. Vector Store (FAISS)

Embeddings are generated using OpenAI's `text-embedding-3-large` (3,072 dimensions) and stored in a FAISS `IndexFlatL2` index for exact L2-distance nearest neighbor search. LangChain's `FAISS` wrapper provides the docstore integration for mapping index positions back to document content and metadata.

In [13]:
import faiss
import numpy as np
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

texts = [doc["text"] for doc in documents]
metadatas = [{
    "passage_id": doc["passage_id"],
    "chunk_index": doc["chunk_index"]} for doc in documents
    ] 

embedding_dim = len(embeddings.embed_query("hello world"))
index = faiss.IndexFlatL2(embedding_dim)

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={}
)

vector_store.add_texts(texts, metadatas=metadatas)
print(f"Index size: {vector_store.index.ntotal}")

Index size: 4658


In [14]:
results = vector_store.similarity_search("Who is Abraham Lincoln?", k=3)
for r in results:
    print(r.page_content[:100], r.metadata)

Abraham Lincoln (February 12, 1809 â April 15, 1865) was the sixteenth President of the United Sta {'passage_id': 278, 'chunk_index': 0}
Young Abraham Lincoln {'passage_id': 288, 'chunk_index': 0}
Abraham Lincoln was born on February 12, 1809, to Thomas Lincoln and Nancy Hanks, two uneducated far {'passage_id': 282, 'chunk_index': 0}


## 5. Custom Retrieval

Instead of using LangChain's built-in retrieval, queries are embedded and searched against the FAISS index directly. This returns raw L2 distances (lower = more similar) along with the document content and metadata.

In [15]:
def retrieve(query, k=5):
    query_vector = np.array([embeddings.embed_query(query)]).astype("float32")
    # Embed the query

    distances, indices = vector_store.index.search(query_vector, k)
    # Search FAISS directly

    results = []
    for i, idx in enumerate(indices[0]):
        doc_id = vector_store.index_to_docstore_id[idx]
        doc = vector_store.docstore.search(doc_id)
        results.append({
            "text": doc.page_content,
            "metadata": doc.metadata,
            "score": distances[0][i],
            "idx": idx,
        })
    return results

In [16]:
results = retrieve("Who is Abraham Lincoln?", k=3)
for r in results:
    print(f"Score: {r['score']:.2f} | {r['text'][:80]}...")

Score: 0.73 | Abraham Lincoln (February 12, 1809 â April 15, 1865) was the sixteenth Preside...
Score: 0.79 | Young Abraham Lincoln...
Score: 0.96 | Abraham Lincoln was born on February 12, 1809, to Thomas Lincoln and Nancy Hanks...


## 6. Cross-Encoder Reranking

Initial retrieval uses a bi-encoder, which embeds query and documents independently — fast but approximate. A cross-encoder processes the query-document pair jointly, producing more accurate relevance scores at the cost of speed.

The pipeline over-retrieves (k=10) with the bi-encoder, then reranks with `cross-encoder/ms-marco-MiniLM-L-6-v2` and keeps the top 3.

In [17]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def retrieve_and_rerank(query, k=10, top_n=3):
    candidates = retrieve(query, k=k)

    pairs = []
    for candidate in candidates:
        pairs.append([query, candidate["text"]])
    
    scores = reranker.predict(pairs)

    for i, candidate in enumerate(candidates):
        candidate["rerank_score"] = float(scores[i])
    
    candidates.sort(key=lambda x: x["rerank_score"], reverse=True)

    return candidates[:top_n]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
results = retrieve_and_rerank("Who is Abraham Lincoln?")
for r in results:
    print(f"Rerank: {r['rerank_score']:.2f} | {r['text'][:80]}...")

Rerank: 9.43 | Abraham Lincoln (February 12, 1809 â April 15, 1865) was the sixteenth Preside...
Rerank: 4.48 | Abraham Lincoln was born on February 12, 1809, to Thomas Lincoln and Nancy Hanks...
Rerank: 3.00 | On November 6, 1860, Lincoln was elected as the 16th President of the United Sta...


## 7. Prompt Construction & RAG Pipeline

The full RAG pipeline: **retrieve → rerank → build prompt → generate answer**.

In [19]:
def build_prompt(query, results):
    context = ""
    for i, r in enumerate(results):
        context += f"[Passage {i+1}]\n{r['text']}\n\n"
    
    system_message = (
        "You are an assistant for question-answering tasks. "
        "Use the following pieces of retrieved context to answer the question. "
        "If you don't know the answer or the context does not contain relevant "
        "information, just say that you don't know. Keep the answer concise."
    )

    prompt = f"{system_message}\n\nContext:\n{context}Question: {query}"
    return prompt

In [20]:
results = retrieve_and_rerank("Was Abraham Lincoln the sixteenth President of the United States?")
prompt = build_prompt("Was Abraham Lincoln the sixteenth President of the United States?", results)
print(prompt)

You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer or the context does not contain relevant information, just say that you don't know. Keep the answer concise.

Context:
[Passage 1]
Abraham Lincoln (February 12, 1809 â April 15, 1865) was the sixteenth President of the United States, serving from March 4, 1861 until his assassination

[Passage 2]
On November 6, 1860, Lincoln was elected as the 16th President of the United States, beating Democrat Stephen A. Douglas, John C. Breckinridge of the Southern Democrats, and John Bell of the new Constitutional Union Party. He was the first Republican president, winning entirely on the strength of his support in the North: he was not even on the ballot in nine states in the South, and won only 2 of 996 counties in the other Southern states

[Passage 3]
and placed on the 'course of ultimate extinction... .'"  Lincoln won the Republican Party nomi

In [ ]:
def rag_query(query, use_hybrid=False):
    if use_hybrid:
        results = retrieve_and_rerank_hybrid(query)
    else:
        results = retrieve_and_rerank(query)

    prompt = build_prompt(query, results)
    response = model.invoke(prompt)
    return response.content

In [22]:
answer = rag_query("Is Yujiang Li a student in SMU?")
print(answer)

I don’t know. The provided context doesn’t mention Yujiang Li or SMU.


## 8. Evaluation

### 8.1 String Matching (Baseline)

In [23]:
def evaluate(num_samples=50):
    correct = 0
    total = num_samples

    for i in range(num_samples):
        question = qa["test"][i]["question"]
        expected = qa["test"][i]["answer"].lower()

        generated = rag_query(question).lower() 

        if expected in generated:
            correct += 1
        else:
            print(f"MISS | Q: {question}")
            print(f"    Expected: {expected} | Got: {generated[:100]}")

    print(f"\nAccuracy: {correct}/{total} = {correct/total:.2%}")


In [24]:
evaluate(50)

MISS | Q: What did The Legal Tender Act of 1862 establish?
    Expected: the united states note, the first paper currency in united states history | Got: the legal tender act of 1862 established the united states note—the first paper currency in u.s. his
MISS | Q: Who suggested Lincoln grow a beard?
    Expected: 11-year-old grace bedell | Got: grace bedell, an 11-year-old girl, suggested that lincoln grow a beard.
MISS | Q: Who was the general in charge at the Battle of Antietam?
    Expected: general mcclellan | Got: general george b. mcclellan was in charge of union forces at the battle of antietam.
MISS | Q: Why did Lincoln issue the Emancipation Proclamation?
    Expected: to free slaves  | Got: lincoln issued the emancipation proclamation to make the abolition of slavery in the rebel (confeder
MISS | Q: What trail did Lincoln use a Farmers' Almanac in? 
    Expected: he defended william "duff" armstrong | Got: i don’t know—the context only says lincoln used a farmers’ almanac in 

**Interpretation:** String matching checks whether the expected answer appears as a substring of the generated answer. The misses reveal a fundamental limitation — many answers are semantically correct but phrased differently:

- Expected: *"11-year-old Grace Bedell"* → Generated: *"Grace Bedell, an 11-year-old girl"* (same content, different word order)
- Expected: *"the United States Note, the first paper currency..."* → Generated uses *"U.S."* instead of *"United States"*

This motivates using an LLM-as-judge for more robust evaluation.

### 8.2 LLM-as-Judge

In [25]:
def llm_judge(question, expected, generated):
    prompt = (
        f"Question: {question}\n"
        f"Expected answer: {expected}\n"
        f"Generated answer: {generated}\n\n"
        "Are these answers semantically equivalent? Reply only 'yes' or 'no'."
    )
    return model.invoke(prompt).content.strip().lower() == "yes"

def llm_evaluate(num_samples=50):
    correct = 0
    
    for i in range(num_samples):
        question = qa["test"][i]["question"]
        expected = qa["test"][i]["answer"]
        generated = rag_query(question)
        
        if llm_judge(question, expected, generated):
            correct += 1
        else:
            print(f"MISS | Q: {question}")
            print(f"  Expected: {expected} | Got: {generated[:100]}")
    
    print(f"\nAccuracy: {correct}/{num_samples} = {correct/num_samples:.2%}")

In [26]:
llm_evaluate(50)

MISS | Q: Who graduated in ecclesiastical law at the early age of 20 and began to practice?
  Expected: Amedeo Avogadro | Got: I don’t know—the context only says “He graduated in ecclesiastical law at the early age of 20 and be
MISS | Q: When did he become a professor?
  Expected: 1820 | Got: He became a professor of physics in **1774** (at the Como high school), and later became a professor
MISS | Q: What happened in 1833?
  Expected: Avogadro had been recalled to Turin university | Got: I don’t know. The provided context mentions events in 1834, 1837, 1839, 1840, 1849, and 1862, but no
MISS | Q: When did he publish a collection?
  Expected: 1733 | Got: I don’t know. The context mentions a collection (“Have Faith in Massachusetts: A Collection of Speec
MISS | Q: Is it true that he published a collection in 1738?
  Expected: No | Got: I don’t know. The context says he published a collection of 316 aurora observations in **1733**, not

Accuracy: 45/50 = 90.00%


**Interpretation:** The LLM-as-judge correctly handles paraphrased answers, giving a more accurate picture of system performance. The remaining misses come from **retrieval failures on vague questions** — e.g., *"What happened in 1833?"* or *"When did he become a professor?"* lack enough context for the retriever to find the correct passage among thousands of candidates. This is a retrieval limitation, not a generation one.

### 8.3 Chunk Size Experiment

In [27]:
def chunk_size_experiment(n_samples):
    global vector_store

    configs = [
        {"chunk_size": 300, "chunk_overlap": 30},
        {"chunk_size": 500, "chunk_overlap": 50},
        {"chunk_size": 500, "chunk_overlap": 100},
        {"chunk_size": 800, "chunk_overlap": 80},
        {"chunk_size": 1000, "chunk_overlap": 100},
    ]

    embedding_dim = len(embeddings.embed_query("hello world"))

    for config in configs:
        documents = []
        for item in filtered:
            chunks = chunk_text(item["text"], chunk_size=config["chunk_size"], chunk_overlap=config["chunk_overlap"])
            for i, chunk in enumerate(chunks):
                documents.append({
                    "text": chunk,
                    "passage_id": item["id"],
                    "chunk_index": i
                })
                
        texts = [doc["text"] for doc in documents]
        metadatas = [{
            "passage_id": doc["passage_id"],
            "chunk_index": doc["chunk_index"]} for doc in documents
            ] 


        index = faiss.IndexFlatL2(embedding_dim)

        vector_store = FAISS(
            embedding_function=embeddings,
            index=index,
            docstore=InMemoryDocstore(),
            index_to_docstore_id={}
        )

        vector_store.add_texts(texts, metadatas=metadatas)
        print(f"\n--- chunk_size={config['chunk_size']}, overlap={config['chunk_overlap']} | {len(documents)} chunks ---")
        llm_evaluate(n_samples)

In [28]:
chunk_size_experiment(50)


--- chunk_size=300, overlap=30 | 7066 chunks ---
MISS | Q: Did Lincoln sign the National Banking Act of 1863?
  Expected: yes | Got: I don’t know. The context says the National Banking Acts of 1863–1865 created the national bank syst
MISS | Q: What trail did Lincoln use a Farmers' Almanac in? 
  Expected: he defended William "Duff" Armstrong | Got: I don’t know. The provided context doesn’t mention Lincoln using a Farmers’ Almanac or any “trail” a
MISS | Q: Who graduated in ecclesiastical law at the early age of 20 and began to practice?
  Expected: Amedeo Avogadro | Got: I don’t know. The provided context says “He graduated in ecclesiastical law at the early age of 20,”
MISS | Q: Did the scientific community not reserve great attention to his theory ?
  Expected: yes | Got: Yes. The context states: “The scientific community did not reserve great attention to his theory,” s
MISS | Q: What happened in 1833?
  Expected: Avogadro had been recalled to Turin university | Got: In 1833, he b

| Config (size/overlap) | Chunks | LLM-Judge Accuracy |
|:-----:|:------:|:------:|
| 300/30 | 7,066 | 84% |
| 500/50 | 4,658 | **90%** |
| 500/100 | 4,794 | 84% |
| 800/80 | 3,608 | 88% |
| 1000/100 | 3,369 | 88% |

**Observations:**
1. **Larger chunks perform slightly better** — the original passages are already short (avg 389 chars), so splitting too aggressively loses context that the LLM needs to generate correct answers.
2. **The same questions fail across all configs** — vague queries like *"What happened in 1833?"* fail regardless of chunk size. This is a retrieval problem (insufficient query context), not a chunking problem.
3. **Differences are small (84–90%)** — chunking is not the primary bottleneck for this dataset.

**Selected config: 500/50** — highest accuracy (90%) with a moderate chunk count.

### 8.4 Comprehensive Evaluation

A full evaluation framework measuring four complementary metrics:
- **Recall@3:** Does any of the top-3 retrieved passages contain the answer? (retrieval quality)
- **MRR** (Mean Reciprocal Rank): How high does the first relevant passage rank? (retrieval ranking)
- **Token F1:** Token-level overlap between expected and generated answers (surface-level accuracy)
- **LLM-as-Judge:** Are the generated and expected answers semantically equivalent? (semantic accuracy)

In [29]:
import time

def judge_relevance(question, expected, passage):
    prompt = (
        f"Question: {question}\n"
        f"Expected answer: {expected}\n"
        f"Passage: {passage}\n\n"
        "Does this passage contain enough information to answer the question correctly? "
        "Reply only 'yes' or 'no'."
    )
    return model.invoke(prompt).content.strip().lower() == "yes"

def compute_f1(expected, generated):
    expected_tokens = expected.lower().split()
    generated_tokens = generated.lower().split()

    common = set(expected_tokens) & set(generated_tokens)

    if len(common) == 0:
        return 0.0

    precision = len(common) / len(generated_tokens)
    recall = len(common) / len(expected_tokens)

    return 2 * precision * recall / (precision + recall)

def full_evaluate(n_samples=None, use_hybrid=False):
    total = n_samples or len(qa["test"]["question"])
    method = "hybrid" if use_hybrid else "regular"

    recall_hits = 0
    mrr_sum = 0
    f1_sum = 0
    judge_correct = 0
    total_time = 0

    for i in range(total):
        question = qa["test"][i]["question"]
        expected = qa["test"][i]["answer"]

        start = time.time()
        if use_hybrid:
            results = retrieve_and_rerank_hybrid(question)
        else:
            results = retrieve_and_rerank(question)
        prompt = build_prompt(question, results)
        generated = model.invoke(prompt).content
        elapsed = time.time() - start
        total_time += elapsed

        # Recall@3: does any retrieved passage contain relevant info?
        # MRR: 1/rank of first relevant passage
        recall_hit = False
        first_relevant_rank = None
        for rank, r in enumerate(results, 1):
            if judge_relevance(question, expected, r["text"]):
                recall_hit = True
                if first_relevant_rank is None:
                    first_relevant_rank = rank
                break
        
        if recall_hit:
            recall_hits += 1
        if first_relevant_rank:
            mrr_sum += 1 / first_relevant_rank
        
        # F1: token overlap
        f1_sum += compute_f1(expected, generated)

        # LLM-as-judge
        if llm_judge(question, expected, generated):
            judge_correct += 1
        
        # Progress
        if (i + 1) % 50 == 0:
            print(f"Progress: {i+1}/{total}")
            
    print(f"\n=== Results ({total} samples, {method}) ===")
    print(f"Recall@3:      {recall_hits/total:.2%}")
    print(f"MRR:           {mrr_sum/total:.4f}")
    print(f"F1:            {f1_sum/total:.4f}")
    print(f"LLM-as-judge:  {judge_correct/total:.2%}")
    print(f"Avg latency:   {total_time/total:.2f}s")

In [30]:
full_evaluate(20)


=== Results (20 samples, regular) ===
Recall@3:      100.00%
MRR:           0.9167
F1:            0.0730
LLM-as-judge:  95.00%
Avg latency:   1.58s


In [31]:
full_evaluate()

Progress: 50/918
Progress: 100/918
Progress: 150/918
Progress: 200/918
Progress: 250/918
Progress: 300/918
Progress: 350/918
Progress: 400/918
Progress: 450/918
Progress: 500/918
Progress: 550/918
Progress: 600/918
Progress: 650/918
Progress: 700/918
Progress: 750/918
Progress: 800/918
Progress: 850/918
Progress: 900/918

=== Results (918 samples, regular) ===
Recall@3:      93.36%
MRR:           0.9054
F1:            0.1012
LLM-as-judge:  83.12%
Avg latency:   1.65s


**Full results on all 918 questions (vector retrieval):**

| Metric | Value | Interpretation |
|:-------|:-----:|:---------------|
| Recall@3 | 93.36% | The retriever finds a relevant passage in its top 3 for 93% of questions |
| MRR | 0.9054 | The first relevant passage appears at rank ~1.1 on average — almost always ranked #1 |
| Token F1 | 0.1012 | Low because generated answers are concise while expected answers use different phrasing |
| LLM-as-Judge | 83.12% | The system correctly answers 83% of all questions end-to-end |
| Avg Latency | 1.65s | Per-query time including embedding, retrieval, reranking, and generation |

**Key insight:** The 10-point gap between Recall@3 (93%) and LLM-as-Judge (83%) reveals that ~10% of failures happen at the **generation stage** — the correct passage is retrieved, but the LLM fails to synthesize the right answer. The remaining ~7% are **retrieval failures**, typically on ambiguous questions that lack sufficient context (e.g., *"When did he become a professor?"* without specifying who).

## 9. Hybrid Search: BM25 + Vector (RRF)

Pure vector search captures semantic similarity but can miss exact keyword matches. BM25 is a classical term-frequency method that excels at lexical matching. **Reciprocal Rank Fusion (RRF)** combines both by assigning each result a score of `1/(k + rank)` from each method, then summing.

The BM25 implementation below is built from scratch, including IDF weighting and document length normalization.

In [36]:
import math
from collections import Counter

class BM25():
    
    def __init__(self, documents, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.docs = [doc.lower().split() for doc in documents]
        self.doc_count = len(self.docs)
        self.avg_dl = sum(len(d) for d in self.docs) / self.doc_count
        # Length normalization: Longer docs get penalized slightly so they don't dominate just by having more words

        self.df = {}
        for doc in self.docs:
            for term in set(doc):
                self.df[term] = self.df.get(term, 0) + 1
        # Document frequency - how many documents contain each term

    def score(self, query, doc_index):
        query_terms = query.lower().split()
        doc = self.docs[doc_index]
        doc_len = len(doc)
        tf = Counter(doc)

        score = 0
        for term in query_terms:
            if term not in self.df:
                continue
            idf = math.log((self.doc_count - self.df[term] + 0.5) / (self.df[term] + 0.5) + 1)
            # IDF (Inverse Document Frequency) - rare words get higher socres.
            term_freq = tf.get(term, 0)
            numerator = term_freq * (self.k1 + 1)
            denominator = term_freq + self.k1 * (1 - self.b + self.b * doc_len / self.avg_dl)
            score += idf * numerator / denominator

        return score 

In [46]:
bm25 = BM25(texts)

# Build a text-to-index mapping for fast lookup
text_to_idx = {t: i for i, t in enumerate(texts)}

def hybrid_retrieve(query, k=20, alpha=0.9):
    # Step 1: Vector search
    vector_candidates = retrieve(query, k=k)
    
    # Step 2: BM25 search — score all docs, take top k
    bm25_scores = [(i, bm25.score(query, i)) for i in range(len(texts))]
    bm25_scores.sort(key=lambda x: x[1], reverse=True)
    bm25_top_k = bm25_scores[:k]
    
    # Step 3: Build RRF scores using text_to_idx for alignment
    rrf_scores = {}  # idx -> score
    
    # Vector RRF
    for rank, candidate in enumerate(vector_candidates, 1):
        idx = text_to_idx.get(candidate["text"])
        if idx is not None:
            rrf_scores[idx] = rrf_scores.get(idx, 0) + alpha * (1 / (60 + rank))
    
    # BM25 RRF
    for rank, (idx, score) in enumerate(bm25_top_k, 1):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + (1 - alpha) * (1 / (60 + rank))
    
    # Step 4: Sort by combined RRF score
    combined = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    
    # Step 5: Return top candidates
    results = []
    for idx, score in combined[:k]:
        results.append({
            "text": texts[idx],
            "metadata": metadatas[idx],
            "rrf_score": score
        })
    
    return results


def retrieve_and_rerank_hybrid(query, k=20, top_n=3, alpha=0.7):
    candidates = hybrid_retrieve(query, k=k, alpha=alpha)

    pairs = []
    for candidate in candidates:
        pairs.append([query, candidate["text"]])
    
    scores = reranker.predict(pairs)

    for i, candidate in enumerate(candidates):
        candidate["rerank_score"] = float(scores[i])
    
    candidates.sort(key=lambda x: x["rerank_score"], reverse=True)

    return candidates[:top_n]

In [47]:
# Test hybrid retrieve + rerank
results = retrieve_and_rerank_hybrid("Was Abraham Lincoln the sixteenth President of the United States?")
for r in results:
    print(f"Rerank: {r['rerank_score']:.2f} | {r['text'][:80]}...")

Rerank: 11.24 | Abraham Lincoln (February 12, 1809 â April 15, 1865) was the sixteenth Preside...
Rerank: -0.45 | *As of 2007, Millard Fillmore remains the last U.S. president who was neither a ...
Rerank: -0.73 | Lincoln's death made the President a martyr to many. Repeated polls of historian...


### Comparison: Regular vs. Hybrid Retrieval

In [48]:
# Compare: regular vs hybrid on 50 samples
print("=== Regular Retrieval ===")
full_evaluate(50, use_hybrid=False)
print("\n=== Hybrid Retrieval (BM25 + Vector) ===")
full_evaluate(50, use_hybrid=True)

=== Regular Retrieval ===
Progress: 50/50

=== Results (50 samples, regular) ===
Recall@3:      96.00%
MRR:           0.9400
F1:            0.0624
LLM-as-judge:  86.00%
Avg latency:   1.61s

=== Hybrid Retrieval (BM25 + Vector) ===
Progress: 50/50

=== Results (50 samples, hybrid) ===
Recall@3:      84.00%
MRR:           0.8300
F1:            0.0665
LLM-as-judge:  76.00%
Avg latency:   1.61s


In [49]:
full_evaluate(use_hybrid=True)

Progress: 50/918
Progress: 100/918
Progress: 150/918
Progress: 200/918
Progress: 250/918
Progress: 300/918
Progress: 350/918
Progress: 400/918
Progress: 450/918
Progress: 500/918
Progress: 550/918
Progress: 600/918
Progress: 650/918
Progress: 700/918
Progress: 750/918
Progress: 800/918
Progress: 850/918
Progress: 900/918

=== Results (918 samples, hybrid) ===
Recall@3:      88.02%
MRR:           0.8571
F1:            0.1034
LLM-as-judge:  80.50%
Avg latency:   1.64s


**Side-by-side results (all 918 questions):**

| Metric | Regular (Vector Only) | Hybrid (BM25 + Vector) |
|:-------|:-----:|:-----:|
| Recall@3 | **93.36%** | 88.02% |
| MRR | **0.9054** | 0.8571 |
| Token F1 | 0.1012 | **0.1034** |
| LLM-as-Judge | **83.12%** | 80.50% |

**Analysis:** Hybrid search underperforms pure vector search on this dataset — an honest and instructive result:

1. **Dataset characteristics favor semantic search** — mini-wikipedia passages are well-formed natural language, and the questions are semantic in nature. OpenAI's `text-embedding-3-large` handles this distribution well without keyword assistance.
2. **BM25 introduces noise** — keyword matching pulls in passages that share surface-level terms but are not semantically relevant, diluting the candidate pool sent to the reranker.
3. **RRF weighting** — even with α=0.7 (70% vector weight), the BM25 signal hurts more than it helps for this data distribution.

Hybrid search is expected to outperform on datasets with keyword-centric queries, entity-heavy questions, or noisier documents where exact term matching provides a complementary signal that embeddings miss.

## Conclusion

This RAG system achieves **83% end-to-end accuracy** (LLM-as-Judge) on 918 questions with **93% Recall@3**, using vector retrieval + cross-encoder reranking + LLM generation. The evaluation framework reveals that retrieval quality is strong, and most remaining errors come from ambiguous questions or generation-stage failures — clear targets for future improvement.